In [0]:
dbutils.widgets.dropdown("environment", "prod", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
from pyspark.sql.functions import col, to_date

config = load_config_from_widget()

# Read bronze table
bronze_table = f"{config['catalog']}.{config['schemas']['bronze']}.orders_raw"

df_clean = (spark.table(bronze_table)
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("total_amount", col("total_amount").cast("decimal(10,2)"))
    .filter(col("quantity") > 0)
    .filter(col("total_amount") > 0)
)

# External table location
external_path = f"{config['storage']['external']['silver']}/orders_clean"
print(f"📂 Writing to: {external_path}")

# Write to external location
df_clean.write.format("delta").mode("overwrite").save(external_path)

# Create external table
silver_schema = config['schemas']['silver']
silver_table = f"{config['catalog']}.{silver_schema}.orders_clean"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{silver_schema}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver_table}
    USING DELTA
    LOCATION '{external_path}'
""")

print(f"✅ {silver_table}: {spark.table(silver_table).count()} records")
print(f"✅ Delta logs created at: {external_path}/_delta_log/")